# Simulations

This notebook assess errors in the GPROF database simulations.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [2]:
from gprof_nn.plotting import set_style
set_style()

## Data

We use collocation between ATMS and GPM CMB and match them with the corresponding GPROF simulator files.


In [23]:
collocations = sorted(list(Path("/edata1/simon/gprof_v8/collocations/combined/atms/gridded/").glob("*.nc")))

In [24]:
collocations

[PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20181028073047.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20181028202107.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20181028210837.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20181028224809.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20190101061434.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20190101070533.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20190101084317.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20190101102446.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20190101120346.nc'),
 PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20190102005101.nc'),


In [4]:
import hdf5plugin
from pyresample.geometry import SwathDefinition

colloc_ind = 224
atms_observations = xr.load_dataset(collocations[colloc_ind], group="input_data")
reference_data = xr.load_dataset(collocations[colloc_ind], group="reference_data")

def get_granule(reference_data: xr.Dataset) -> int:
    """
    Extract granule from collocation attributes.

    Args:
        reference_data: An xarray.Dataset containing the collocation reference data

    Return:
        The granule number as an integer.
    """
    l1c_file = reference_data.attrs["input_files"].replace(",,,", ";").replace(",", "").split(";")[0]
    granule = int(l1c_file.split(".")[-3])
    return granule

granule = get_granule(reference_data)
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)

## GPROF-NN 1D Training Data

We use the GPROF-NN 1D training data to extract the simulated and Satformer observations.

In [14]:
from datetime import datetime
from typing import Optional
from pansat import TimeRange
from pansat.time import to_datetime64
from pansat.utils import resample_data

training_files = sorted(list(Path("/edata2/simon/gprof_v8/training_data/atms/sim/1d").glob("*.nc")))
start_times = []
end_times = []

for path in training_files:
    parts = path.name.split('_')
    start_time = datetime.strptime(parts[1], "%Y%m%d%H%M%S")
    start_times.append(to_datetime64(start_time))
    end_time = datetime.strptime(parts[2][:-3], "%Y%m%d%H%M%S")
    end_times.append(to_datetime64(end_time))

start_times = np.array(start_times)
end_times = np.array(end_times)


def find_training_file(time: np.datetime64) -> Optional[Path]:
    """
    Find GPROF-NN 1D training file for a given time.

    Args:
        time: The time for which to find the corresponding training file.

    Return:
        A path object pointing to the training file covering the given time. If no such
        time is available, 'None' is returned.
    """
    print(start_times, time)
    ind = np.searchsorted(start_times, time)
    end_time = end_times[ind - 1]
    if end_time < time:
        return None
    return training_files[ind - 1]

    
def load_and_resample_training_data(amsr2_observations, area) -> Optional[xr.Dataset]:
    """
    Load and resample GPROF-NN 1D training data for a given AMSR2/GPM collocation.

    Args:
        amsr2_observations: An xarray.Dataset containing the AMSR2 observations.
        area: A pyresample area definition defining the grid to which to resample the simulator data.

    Return:
        An xarray.Dataset containing the resample simulator data.
    """
    mean_time = amsr2_observations.scan_time.mean().data

    training_file = find_training_file(mean_time)
    print(mean_time, training_file)
    training_data = xr.load_dataset(training_file, engine="h5netcdf")
    
    lons, lats = area.get_lonlats()
    lon_min = lons.min()
    lon_max = lons.max()
    lat_min = lats.min()
    lat_max = lats.max()
    sample_mask = (
        (lon_min <= training_data.longitude.data) * (training_data.longitude.data <= lon_max) *
        (lat_min <= training_data.latitude.data) * (training_data.latitude.data <= lat_max)
    )
    training_data = training_data[{"samples": sample_mask}].rename({"simulated_brightness_temperatures": "simulated_tbs"})

    training_data_r = resample_data(training_data, area, new_dims=("latitude", "longitude"), radius_of_influence=15e3)
    return training_data_r

## Case study

In [9]:
%env PANSAT_PASSWORD=not_a_secret

env: PANSAT_PASSWORD=not_a_secret


In [22]:
collocations[1]

PosixPath('/edata1/simon/gprof_v8/collocations/combined/atms/gridded/cmb_atms_20181028202107.nc')

In [18]:
import hdf5plugin
from pyresample.geometry import SwathDefinition

colloc_ind = 5
colloc = collocations[colloc_ind]

atms_observations = xr.load_dataset(colloc, group="input_data", engine="h5netcdf")
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)
reference_data = xr.load_dataset(colloc, group="reference_data")
    
training_data_r = load_and_resample_training_data(atms_observations, area)

['2018-10-06T00:43:40.000000000' '2018-10-06T02:16:15.000000000'
 '2018-10-06T03:48:51.000000000' '2018-10-06T05:21:27.000000000'
 '2018-10-06T06:54:01.000000000' '2018-10-06T08:26:36.000000000'
 '2018-10-06T09:59:12.000000000' '2018-10-06T11:31:46.000000000'
 '2018-10-06T13:04:21.000000000' '2018-10-06T14:36:57.000000000'
 '2018-10-06T16:09:31.000000000' '2018-10-06T17:42:07.000000000'
 '2018-10-06T19:14:42.000000000' '2018-10-06T20:47:16.000000000'
 '2018-10-06T22:19:52.000000000' '2018-10-06T23:52:28.000000000'
 '2018-10-07T01:25:01.000000000' '2018-10-07T02:57:37.000000000'
 '2018-10-07T04:30:13.000000000' '2018-10-07T06:02:47.000000000'
 '2018-10-07T07:35:22.000000000' '2018-10-07T09:07:58.000000000'
 '2018-10-07T10:40:32.000000000' '2018-10-07T12:13:07.000000000'
 '2018-10-07T13:45:43.000000000' '2018-10-07T15:18:17.000000000'
 '2018-10-07T18:23:28.000000000' '2018-10-07T19:56:02.000000000'
 '2018-10-07T21:28:38.000000000' '2018-10-07T23:01:14.000000000'
 '2018-10-08T00:33:47.000

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
sim_data_r.satformer_tbs.shape

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 4

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1], np.mean((tbs_sim[valid] - tbs_ref[valid]) ** 2))
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs_rand.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1], np.mean((tbs[valid] - tbs_ref[valid]) ** 2))

ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 3

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1], np.mean((tbs_sim[valid] - tbs_ref[valid]) ** 2))
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1], np.mean((tbs[valid] - tbs_ref[valid]) ** 2))

ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 4

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1], np.mean((tbs_sim[valid] - tbs_ref[valid]) ** 2))
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1], np.mean((tbs[valid] - tbs_ref[valid]) ** 2))

ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 0

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 4

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

## Run Satformer

Below we produce synthetic Tbs using the Satformer model trained on Tb collocations between GMI, ATMS and AMSR2

In [ ]:
from pansat import TimeRange
from pansat.catalog import Index
from pansat.products.satellite.gpm import l1c_r_gpm_gmi, l1c_noaa20_atms
from pansat.environment import get_index
from pansat.catalog.index import find_matches
from gprof_nn.data.pretraining import InputLoader
from pytorch_retrieve.architectures import load_and_compile_model, load_model
from pytorch_retrieve import InferenceConfig
from pytorch_retrieve.config import RetrievalOutputConfig
from pytorch_retrieve.inference import run_inference
from pytorch_retrieve.retrieval_output import ExpectedValue

model = load_model("../satformer_avg_pool-v7.ckpt").eval()

output_config = RetrievalOutputConfig(model.output_config["output_observations"], "ExpectedValue", {})
retrieval_output = {"output_observations": {"output_observations": output_config}}
inference_config = InferenceConfig(tile_size=128, spatial_overlap=32, retrieval_output=retrieval_output)

def run_satformer(input_data, area) -> xr.Dataset:
    """
    Run satformer for given collocation scene and resample the results.
    """
    time_range = TimeRange(input_data.scan_time.mean().item())
    gmi_recs = l1c_r_gpm_gmi.get(time_range=time_range)
    atms_recs = l1c_noaa20_atms.get(time_range=time_range)
    gmi_index = Index.index(l1c_r_gpm_gmi, gmi_recs)
    atms_index = Index.index(l1c_noaa20_atms, atms_recs)
    matches = find_matches(gmi_index, atms_index)
    input_loader = InputLoader(matches)
    inpt, fname, aux = input_loader.load_data(0)
    
    results = run_inference(
        model,
        input_loader,
        inference_config,
        exclude_from_tiling=[
            "input_observation_mask",
            "two_meter_temperature_mask",
            "total_column_water_vapor_mask",
            "land_fraction_mask",
            "ice_fraction_mask",
            "leaf_area_index_mask",
            "elevation_mask",
            "ir_observations_mask",
        ],
    )
    
    results[0]["latitude"] = (("x", "y"), aux["latitude"].data)
    results[0]["longitude"] = (("x", "y"), aux["longitude"].data)
    invalid = np.isnan(inpt["output_observation_props"].numpy()).any(1)[0, 0]
    results[0]["output_observations"].data[..., invalid] = np.nan
    results_r = resample_data(results[0].transpose("x", "y", ...), area, radius_of_influence=15e3)

    return results_r

In [ ]:
sf_data_r = run_satformer(atms_observations, area)

In [ ]:
sf_data_r

In [ ]:
plt.pcolormesh(sf_data_r.output_observations[..., 7])
plt.colorbar()

## Collect data from collocations

Below, we iterate over all collocations and extract data from pixels with valid simulator data.

In [ ]:
from tqdm import tqdm
tbs_actual = []
tbs_sim = []
tbs_sim_bias = []
tbs_sf = []
eia = []
surface_type = []
surface_precip = []

for colloc in tqdm(np.random.permutation(collocations)[:40]):
    
    atms_observations = xr.load_dataset(colloc, group="input_data")
    lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
    area = SwathDefinition(lats=lats, lons=lons)

    reference_data = xr.load_dataset(colloc, group="reference_data")
    granule = get_granule(reference_data)
    
    sim_data_r = load_and_resample_sim_data(granule, atms_observations, area)
    valid = (
        (atms_observations.observations_gprof.data > -1000).any(axis=-1) *
        (sim_data_r.simulated_tbs.data > -1000).any(axis=-1) *
        (sim_data_r.brightness_temperature_biases.data > -1000).any(axis=-1)
    )
    
    tbs_actual.append(atms_observations.observations_gprof.data[valid])
    tbs_sim.append(sim_data_r.simulated_tbs.data[valid])
    tbs_sim_bias.append(sim_data_r.brightness_temperature_biases.data[valid])
    tbs_sf.append(sim_data_r.satformer_tbs.data[valid])
    eia.append(atms_observations.earth_incidence_angle.data[valid, 0])
    surface_type.append(sim_data_r.surface_type.data[valid])
    surface_precip.append(sim_data_r.surface_precip.data[valid])


In [ ]:
atms_observations.observations_gprof

In [ ]:
results = xr.Dataset({
    "tbs_actual":  (("samples", "channels"), np.concatenate(tbs_actual)),
    "tbs_sim": (("samples", "channels"), np.concatenate(tbs_sim)),
    "tbs_sim_bias": (("samples", "channels"), np.concatenate(tbs_sim_bias)),
    "tbs_sf": (("samples", "channels"), np.concatenate(tbs_sf)),
    "eia": (("samples"), np.concatenate(eia)),
    "surface_type":  (("samples",), np.concatenate(surface_type)),
})


In [ ]:
results["tbs_actual"]

In [ ]:
from matplotlib.gridspec import GridSpec

CHANNELS = [
    "89 GHz",
    "164 GHz",
    "183 +/- 1 GHz",
    "183 +/- 3 GHz",
    "183 +/- 7 GHz",
]

def make_scater_plots(results):
    fig = plt.figure(figsize=(20, 25))
    gs = GridSpec(5, 5, width_ratios=[0.3, 1.0, 1.0, 1.0, 1.0])
    
    
    for chan in range(5):
        
        ax = fig.add_subplot(gs[chan, 0])
        ax.set_axis_off()
        ax.text(0, 0, CHANNELS[chan], rotation=90, ha="center", va="center")
        ax.set_ylim(-2, 2)
        
        tbs_act = results["tbs_actual"].data[..., chan]
        tbs_sim = results["tbs_sim"].data[..., chan]
        tbs_bias = results["tbs_sim_bias"].data[..., chan]
        tbs_sf = results["tbs_sf"].data[..., chan]
        eia = results["eia"].data
        
        #
        # Simulated TBS
        #
        
        ax = fig.add_subplot(gs[chan, 1])
        if chan == 0:
            ax.set_title("Simulated", loc="center")
        bins = np.linspace(tbs_act.min(), tbs_act.max())
        tbs = tbs_sim
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")

        ax.set_ylabel("Simulated $T_b$ [K]")
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Bias-corrected, simulated TBs
        #
        
        ax = fig.add_subplot(gs[chan, 2])
        if chan == 0:
            ax.set_title("Simulated - Bias", loc="center")
        tbs = tbs_sim - tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # EIA adapted bias correction
        #
        
        ax = fig.add_subplot(gs[chan, 3])
        if chan == 0:
            ax.set_title(r"Simulated - $\frac{\cos(\theta_{\text{GMI}})}{\cos(\theta)}$ Bias", loc="center")
        tbs = tbs_sim - np.cos(np.deg2rad(48.0)) / np.cos(np.deg2rad(eia)) * tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Satformer results
        #
        
        ax = fig.add_subplot(gs[chan, 4])
        if chan == 0:
            ax.set_title(r"Satformer", loc="center")
        
        tbs = tbs_sf
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
                                 
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        #ax.text(0.1 * x[0], 0.9 * x[-1], f"Bias:  {bias:.2f}\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}")
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)

    return fig, ax

In [ ]:
100

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

## Test simulations

In [ ]:
time_range = TimeRange("2020-06-03", "2020-06-04")
gmi_recs = l1c_r_gpm_gmi.get(time_range=time_range)
atms_recs = l1c_noaa20_atms.get(time_range=time_range)
gmi_index = Index.index(l1c_r_gpm_gmi, gmi_recs)
atms_index = Index.index(l1c_noaa20_atms, atms_recs)
matches = find_matches(gmi_index, atms_index)
input_loader = InputLoader(matches)

In [ ]:
inpt, fname, aux = input_loader.load_data(1)

In [ ]:
inpt["observations"].shape

In [ ]:
plt.pcolormesh(inpt["observations"][0, 2, 6])

In [ ]:
inpt.keys()

In [ ]:
tile = {name: tensor[..., 350:478, 20:148] for name, tensor in inpt.items()}
for name, tensor in inpt.items():
    if name.endswith("_mask"):
        tile[name] = inpt[name]
tbs_targ = aux["target_observations"].data[..., 64:128, 64:128]

In [ ]:
tile["observations"].shape

In [ ]:
plt.imshow(tile["observations"][0, 2, 0])

In [ ]:
import torch

with torch.no_grad():
    y_pred = model(tile)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

In [ ]:
from copy import deepcopy

props = tile["output_observation_props"].clone()
props[:, :, 1:] = props[:, :, :1]
props[0, 3, :] = torch.tensor(np.linspace(1.75, 5.2, 9))[..., None, None]
props[0, 4, :] = torch.tensor(np.linspace(416e3, 800e3, 9))[..., None, None]

tile_bw = deepcopy(tile)
tile_bw["output_observation_props"] = props

In [ ]:

with torch.no_grad():
    y_pred = model(tile_bw)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

## Beam width

In [ ]:
def simulate_beam_width(beam_width):
    
    props = tile["output_observation_props"].clone()[:, :, :1]
    props[0, 3, :] = beam_width
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"89 GHz, beam width = {beam_width:.2f} deg.")
    plt.imshow(y_pred[0], vmin=160, vmax=240)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)


In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

interact(simulate_beam_width, beam_width=(1.5, 5.2))


## WV-channel offset

In [ ]:
def simulate_offset(offset):
    
    props = tile["output_observation_props"].clone()[:, :, -1:]
    props[0, 1, :] = offset
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"183 +/- {offset:.2f} GHz")
    plt.imshow(y_pred[0], vmin=240, vmax=270)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_offset, offset=(1.0, 7.0))

## EIA

In [ ]:
def simulate_eia(eia):
    
    props = tile["output_observation_props"].clone()[:, :, [-5]]
    props[0, -2, :] += eia
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(rf"183 +/- 7 GHz, $\Delta\theta = ${eia:.2f}")
    plt.imshow(y_pred[0], vmin=220, vmax=280)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_eia, eia=(-20.0, 20.0))

In [ ]:
plt.imshow(props[0, -2, -1])
plt.colorbar()

## Run simul